<a href="https://colab.research.google.com/github/Caleb-John1603/BIS_LabCodeDump/blob/main/Ant%20Colony%20optimisation%201BM24CS095.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import random

class ForestFireACO:
    def __init__(self, alpha=1.0, beta=2.0, rho=0.2, Q=5.0):
        self.num_nodes = 5
        self.source = 0
        self.base_station = 4
        self.alpha = alpha
        self.beta = beta
        self.rho = rho
        self.Q = Q

        self.adj_matrix = np.array([
         [0, 0, 1, 1, 0],
         [0, 0, 0, 1, 0],
         [1, 0, 0, 0, 1],
         [1, 1, 0, 0, 0],
         [0, 0, 1, 0, 1]
        ])

        self.battery = np.array([100.0, 85.0, 95.0, 40.0, 100.0])

        self.energy_cost = np.array([
            [np.inf, 2.5,    1.8,    np.inf, np.inf],
            [2.5,    np.inf, np.inf, 3.0,    np.inf],
            [1.8,    np.inf, np.inf, 2.0,    1.5],
            [np.inf, 3.0,    2.0,    np.inf, 2.8],
            [np.inf, np.inf, 1.5,    2.8,    np.inf]
        ])

        self.link_quality = np.array([
            [0.0, 0.8, 0.9, 0.0, 0.0],
            [0.8, 0.0, 0.0, 0.7, 0.0],
            [0.9, 0.0, 0.0, 0.6, 0.95],
            [0.0, 0.7, 0.6, 0.0, 0.5],
            [0.0, 0.0, 0.95, 0.5, 0.0]
        ])

        self.pheromones = np.zeros((5, 5))
        for i in range(5):
            for j in range(5):
                if self.adj_matrix[i][j] == 1:
                    self.pheromones[i][j] = 1.0

    def calculate_heuristic(self, curr, neighbor):
        lq = self.link_quality[curr][neighbor]
        bat = self.battery[neighbor]
        ec = self.energy_cost[curr][neighbor]
        return (lq * bat) / ec

    def run_optimization(self, num_ants=10, max_iterations=30):
        best_path = None
        best_fitness = -1.0

        print("--- Optimization Initiated ---")
        for iteration in range(max_iterations):
            all_paths = []
            for ant in range(num_ants):
                current = self.source
                path = [current]
                visited = {current}
                failed = False

                while current != self.base_station:
                    neighbors = [n for n in range(self.num_nodes) if self.adj_matrix[current][n] == 1 and n not in visited]
                    if not neighbors:
                        failed = True
                        break

                    probabilities = []
                    for n in neighbors:
                        tau = self.pheromones[current][n]
                        eta = self.calculate_heuristic(current, n)
                        prob = (tau ** self.alpha) * (eta ** self.beta)
                        probabilities.append(prob)

                    sum_prob = sum(probabilities)
                    if sum_prob == 0:
                        probabilities = [1.0 / len(neighbors)] * len(neighbors)
                    else:
                        probabilities = [p / sum_prob for p in probabilities]

                    next_node = random.choices(neighbors, weights=probabilities, k=1)[0]
                    path.append(next_node)
                    visited.add(next_node)
                    current = next_node

                if not failed:
                    all_paths.append(path)

            for i in range(5):
                for j in range(5):
                    if self.adj_matrix[i][j] == 1:
                        self.pheromones[i][j] *= (1 - self.rho)

            for path in all_paths:
                total_energy = sum(self.energy_cost[path[i]][path[i+1]] for i in range(len(path)-1))
                min_battery = min(self.battery[node] for node in path)
                fitness = min_battery / total_energy

                if fitness > best_fitness:
                    best_fitness = fitness
                    best_path = path

                for i in range(len(path) - 1):
                    u, v = path[i], path[i+1]
                    self.pheromones[u][v] += self.Q * fitness
                    self.pheromones[v][u] += self.Q * fitness

        self.display_network_state(best_path, best_fitness)

    def display_network_state(self, best_path, best_fitness):
        print("\n================== ENVIRONMENT STATE ==================")
        print(f"Source Node (Fire Detected): {self.source}")
        print(f"Destination Base Station:    {self.base_station}")
        print("\n Sensor Connection Matrix (Adjacency Graph):")
        print(self.adj_matrix)
        print("\n Finalized Pheromone Trails List (Active Edges):")
        print(f"{'Edge (u, v)':<12} | {'Pheromone Value':<15}")
        print("-" * 32)
        seen_edges = set()
        for i in range(5):
            for j in range(5):
                if self.adj_matrix[i][j] == 1 and (j, i) not in seen_edges:
                    seen_edges.add((i, j))
                    print(f"({i}, {j}){'':<7} | {self.pheromones[i][j]:.4f}")
        print("\n Optimization Outcome:")
        if best_path:
            print(f"➔ Optimal Discovered Path:  {' ➔ '.join(map(str, best_path))}")
            print(f"➔ Highest Path Fitness:     {best_fitness:.4f}")
        else:
            print("➔ Failure: No valid path discovered to base station.")
        print("=======================================================")

if __name__ == "__main__":
    random.seed(10)
    network_aco = ForestFireACO()
    network_aco.run_optimization(num_ants=8, max_iterations=25)


--- Optimization Initiated ---

================== ENVIRONMENT STATE ==================
Source Node (Fire Detected): 0
Destination Base Station:    4

 Sensor Connection Matrix (Adjacency Graph):
[[0 0 1 1 0]
 [0 0 0 1 0]
 [1 0 0 0 1]
 [1 1 0 0 0]
 [0 0 1 0 1]]

 Finalized Pheromone Trails List (Active Edges):
Edge (u, v)  | Pheromone Value
--------------------------------
(0, 2)        | 5735.8280
(0, 3)        | 0.0038
(1, 3)        | 0.0038
(2, 4)        | 5735.8280
(4, 4)        | 0.0038

 Optimization Outcome:
➔ Optimal Discovered Path:  0 ➔ 2 ➔ 4
➔ Highest Path Fitness:     28.7879
